# STAC AI Workshop · NYC · 21 October 2026
## 1 · Fixed RAG versus an agent: which component does what?
**Matthew Dixon, PhD, FRM · BeliefLens**

Run each cell in order. Every input and output appears in a readable box.

**Our question:** Does Harbor Metals have comfortable covenant headroom?

| Workflow | Who retrieves documents? | Who calls the calculator? |
|---|---|---|
| Fixed RAG | Application code | Nobody |
| Fixed RAG + calculator | Application code | Application code |
| Agent | Model requests; application executes | Model requests; application executes |

**RAG means retrieval augmented generation:** send a query to a document search service, receive matching text, then include that text with the question sent to the LLM.

By the end, you will be able to locate the retrieval query, specify which documents can be searched, call the calculator, and inspect an agent's choices.

The default is a scripted synthetic LLM. Its implementation is in `architecture_support.py`; it illustrates the interface and does not measure real model ability.

## Open and run this notebook
[Open in Google Colab](https://colab.research.google.com/github/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/01_architectures.ipynb) · [View on GitHub](https://github.com/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/01_architectures.ipynb)

1. Run the **COLAB_BOOTSTRAP** cell first. It downloads public helpers and saved fixtures when using Colab; it does nothing in local JupyterLab.
2. Run the named code cells from top to bottom. Leave live, fresh-inference and GPU switches off initially.
3. References such as **C03:5** mean line 5 inside code cell C03. The guide quotes the exact code so you can find it without editor line numbers. Enable line numbers in your editor preferences if available.
4. Read **Answer checks** below to compare the default result. Edited inputs and live runs can differ; the stated conditions explain what should remain true.

Saved results, constructed examples, rule checks and fresh model calls are identified separately. Do not save API keys in cells or outputs.

In [10]:
# COLAB_BOOTSTRAP · download the public workshop helpers and saved fixtures.
import sys
if "google.colab" in sys.modules:
    from pathlib import Path
    import os, subprocess
    workshop_root = Path("/content/STAC_AI_Bootcamp")
    if not (workshop_root / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mfrdixon/STAC_AI_Bootcamp.git", str(workshop_root)], check=True)
    os.chdir(workshop_root / "notebooks")
    sys.path.insert(0, str(workshop_root))
    sys.path.insert(0, str(workshop_root / "notebooks"))
    print("Workshop helpers and saved fixtures ready. No model API call was made.")

Workshop helpers and saved fixtures ready. No model API call was made.


### Open in your browser
[Open this notebook in Google Colab](https://colab.research.google.com/github/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/01_architectures.ipynb), or choose **File → Open notebook → GitHub** in Colab and paste its GitHub address. Choose **File → Save a copy in Drive** for your own copy.

Local JupyterLab users: keep `architecture_support.py` beside this notebook. Colab users: the setup below downloads that helper from this public repository. The helper contains the scripted model and provider adapter; the workflow itself stays visible below.

### Following line references
Code cells are named **S1, A1, B1, C1 and D1** in their first-line comments. A reference such as **C1:5** means line 5 inside code cell C1; it does not mean execution counter `[5]`. Enable line numbers in your editor preferences if available. The guide beside each cell also quotes the exact line, so you can find it without editor line numbers.

**First run:** leave `PROVIDER="synthetic"`; no API key is needed. Run the helper setup, then S1, A1, B1, C1 and D1 in order.

In [11]:
# Colab companion-file setup. No model call and no API key required.
import sys
if "google.colab" in sys.modules:
    from pathlib import Path
    from urllib.request import urlretrieve
    if not Path("architecture_support.py").exists():
        urlretrieve("https://raw.githubusercontent.com/mfrdixon/STAC_AI_Bootcamp/main/notebooks/architecture_support.py", "architecture_support.py")

In [12]:
# S1 · Setup
from architecture_support import (
    DOCUMENTS, INITIAL_DOCUMENTS, QUESTION, DocumentSearch,
    ModelSession, calculate_leverage, financial_inputs, restricted_cash, panel,
)

# These are the documents available to this exercise.
rag = DocumentSearch(DOCUMENTS)
panel("Available documents", "\n\n".join(
    f"{id}: {doc['text']}" for id, doc in DOCUMENTS.items()
))
panel("Question sent to the model", QUESTION)

# Default: no key or network needed.
PROVIDER = "synthetic"  # optional: "openai" or "together"
MODEL = ""              # enter a tool-capable model ID for live mode

def new_model():
    return ModelSession(provider=PROVIDER, model=MODEL)


### A · Fixed RAG: retrieve → send documents → read answer
The **application** specifies both the query and the documents to search. Here it searches the filing and covenant. The annex is available but is outside this initial search.

`documents` is the retrieval output. `add_context` sends that returned text alongside the question to the model. No calculator is called.

**Before running:** find `query` and `document_ids` below. Those are the application's retrieval instructions.

**After running:** look for FILING and COVENANT in the returned passages. The application puts those passages in the model input. The model does not choose another search in this workflow.

**Code cell A1 — where to look**

| Line | Code | What it does |
|---|---|---|
| A1:2 | `query = "Harbor Metals debt cash EBITDA covenant"` | The application specifies the search query. |
| A1:3 | `documents = rag.search(query=query, document_ids=INITIAL_DOCUMENTS)` | The application calls the retrieval tool; its result is stored in documents. |
| A1:7 | `model.add_context(documents)` | The returned passages are supplied to the model. |
| A1:8 | `answer = model.next()` | The model is asked to answer; no tools are enabled for this call. |


In [13]:
# A1 · Fixed RAG
query = "Harbor Metals debt cash EBITDA covenant"
# documents = rag.search(query=query, document_ids=INITIAL_DOCUMENTS)
documents = rag.search(query=query, document_ids=['Filings'])
panel("RAG output · text returned for the query", documents)

model = new_model()
model.add_context(documents)
answer = model.next()
panel("Model output · fixed RAG", answer)


### Check A1: expected offline result
- The query is set by application code. The returned sources are **FILING and COVENANT**.
- **ANNEX_A is not yet supplied.** No calculator function runs in A1.
- The scripted model states that current headroom still needs verification; do not treat its prose as an executed calculation.
- RAG is visible at `rag.search(...)` followed by `model.add_context(documents)`.

### B · Fixed RAG + calculator: retrieve → calculate → send both → read answer
The application extracts figures from the filing, supplies the calculator arguments, and sends its result to the model. `financial_inputs` is a small parser for this controlled document format.

**Inspect the cash input:** this simple pipeline uses all $15m of reported cash. The covenant refers to an annex. Does a correct calculation establish the right covenant ratio?

**Before running:** locate the actual function call `calculate_leverage(**inputs)`. The application runs it before asking for the answer.

**After running:** look for cash **15** and leverage **3.00×**. This arithmetic uses reported cash; it has not yet resolved the annex's restriction.

**Code cell B1 — where to look**

| Line | Code | What it does |
|---|---|---|
| B1:3 | `inputs = financial_inputs(documents)` | Application code extracts calculator inputs from the retrieved filing. |
| B1:6 | `calculation = calculate_leverage(**inputs)` | This is the actual Python calculator call. |
| B1:10 | `model.add_context(documents, calculation)` | Both the source passages and calculator result enter the model context. |


In [14]:
# B1 · Fixed RAG plus calculator
documents = rag.search(query=query, document_ids=INITIAL_DOCUMENTS)
inputs = financial_inputs(documents)
panel("Calculator input · chosen by application code", inputs)

calculation = calculate_leverage(**inputs)
panel("Calculator output", calculation)

model = new_model()
model.add_context(documents, calculation)
answer = model.next()
panel("Model output · fixed RAG + calculator", answer)


In [15]:
documents

{'query': 'Harbor Metals debt cash EBITDA covenant',
 'searched': ['FILING', 'COVENANT'],
 'documents': [{'id': 'FILING',
   'text': 'Harbor Metals: debt 75 million; cash 15 million; EBITDA 20 million.'},
  {'id': 'COVENANT',
   'text': 'Ceiling 3.5 times. Exclude restricted cash specified in Annex A.'}]}

In [16]:
inputs

{'debt': 75.0, 'eligible_cash': 15.0, 'ebitda': 20.0}

In [17]:
INITIAL_DOCUMENTS

['FILING', 'COVENANT']

### Check B1: expected offline result
- Calculator inputs: **debt 75, eligible_cash 15, ebitda 20**.
- Executed calculation: **(75 − 15) / 20 = 3.00×**.
- The arithmetic is correct for those inputs, but the cash eligibility has not been resolved from the annex. Correct arithmetic alone does not establish the contractual ratio.

### C · Agent: ask model → execute its requested tool → return result → repeat
Here the model initially receives the question and two tool definitions: `retrieve` and `calculate_leverage`. It chooses a retrieval query, document IDs and calculator arguments. Python executes each request and sends the result back.

The loop below is the workflow. Follow the numbered boxes: **model request → tool result → next model request**. The six-turn limit bounds the investigation.

**Before running:** notice that the loop does not specify “search the annex next.” It asks the model for its next request.

**After running:** follow the boxes in order: initial retrieval → annex request → calculator with eligible cash **10** → answer. Python executes the requests. The default model is scripted to illustrate this sequence; live models may behave differently.

**Code cell C1 — where to look**

| Line | Code | What it does |
|---|---|---|
| C1:5 | `response = model.next(allow_tools=True)` | THIS asks the model for its next request or final answer. Tools are enabled. |
| C1:9 | `for call in response.calls:` | The application inspects the structured requests returned by the model. |
| C1:11 | `result = rag.search(**call.arguments)` | Python executes a requested retrieval. |
| C1:13 | `result = calculate_leverage(**call.arguments)` | Python executes a requested calculation. |
| C1:18 | `model.observe(call, result)` | The application returns the tool result to the model before the next loop iteration. |


In [20]:
# C1 · Agent tool loop
model = new_model()
agent_calculation = None
for step in range(1, 7):
    response = model.next(allow_tools=True)
    panel(f"Step {step} · model output", response)
    if not response.calls:
        break
    for call in response.calls:
        if call.name == "retrieve":
            result = rag.search(**call.arguments)
        elif call.name == "calculate_leverage":
            result = calculate_leverage(**call.arguments)
            agent_calculation = result
        else:
            raise ValueError("Tool not permitted")
        panel(f"Step {step} · {call.name} result", result)
        model.observe(call, result)
else:
    panel("Stop", "Tool budget reached; review required.")


# An answer is not proof that the model used the calculator.
if agent_calculation is None:
    panel("Verification", "NOT VERIFIED: no calculator call completed. Review the answer.")
else:
    panel("Executed calculator result", agent_calculation)
    panel("Limit of this check", "Confirms tool execution, not that every sentence agrees with the result.")

### Check C1: expected offline result
1. Model requests the initial filing/covenant search.
2. Model requests **ANNEX_A** after the covenant refers to it.
3. Model requests the calculator with **debt 75, eligible_cash 10, ebitda 20**.
4. Python returns **3.25×**; the model then answers.

The scripted model illustrates the tool loop. Live models can choose different requests or skip a step. The verification box flags a missing calculator result; it does not certify every final sentence.

### Try one change: give the fixed workflow an annex step
An agent is not required when you can specify the needed steps in code. Here the application explicitly retrieves Annex A and excludes restricted cash before calculating.

Compare the returned ratio with the earlier **3.00×**. The result should be **3.25×**. A current calculation still leaves scenario risk and approval unresolved.

**Why this matters:** the agent has no exclusive ability to find the annex. A fixed workflow can explicitly retrieve it too. The comparison concerns where the next-step choice is made, not proof that an agent is more accurate.

**Code cell D1 — where to look**

| Line | Code | What it does |
|---|---|---|
| D1:2 | `annex = rag.search(query="restricted cash", document_ids=["ANNEX_A"])` | Application code prescribes an additional annex search. |
| D1:9 | `inputs["eligible_cash"] -= restricted_cash(annex)  # read the returned annex` | The application deducts the restricted cash stated in the annex. |
| D1:11 | `panel("Corrected calculator output", calculate_leverage(**inputs))` | The application explicitly calls the calculator with corrected inputs. |


In [21]:
# D1 · Fixed annex branch
annex = rag.search(query="restricted cash", document_ids=["ANNEX_A"])
panel("Extra RAG query and returned annex", annex)

if not annex["documents"]:
    panel("Decision", "HOLD: eligible cash cannot be established without the annex.")
else:
    inputs = financial_inputs(documents)
    inputs["eligible_cash"] -= restricted_cash(annex)  # read the returned annex
    panel("Corrected calculator input", inputs)
    panel("Corrected calculator output", calculate_leverage(**inputs))


### Check D1: expected offline result
- The fixed branch retrieves the annex and deducts **5** of restricted cash.
- Eligible cash becomes **10**, and the calculator returns **3.25×**, matching the scripted agent.
- **Conclusion:** a fixed workflow can retrieve the annex too when its code specifies that step. The difference is which component selects the sequence, not exclusive access to a tool.

### Class exercise — make one controlled change
1. In **A1:3**, replace `document_ids=INITIAL_DOCUMENTS` with `document_ids=["FILING"]`. Rerun A1. Which source disappears?
2. Restore that line. In a new cell immediately before C1, run `rag.documents.pop("ANNEX_A", None)`. Rerun C1. What does the annex request return? Does a calculator result exist?
3. Restore the store by rerunning **S1:8** (`rag = DocumentSearch(DOCUMENTS)`), then rerun C1. What cash argument reaches the calculator?
4. Find **C1:5**. Explain why it asks for a next step rather than prescribing the annex step in application code.

### Answers to check your work
1. **COVENANT disappears** from A1's retrieval output and supplied context. The filing remains.
2. The model requests the annex, but retrieval returns no annex passage. The scripted path withholds a supported calculation; the verification reports **no completed calculator call**. A live model may behave differently—inspect the trace rather than assume it handled the gap.
3. The cash argument is **10**, and executed leverage is **3.25×**.
4. `model.next(allow_tools=True)` returns what the model chooses to request from the allowed tools. Python executes the requests and returns results through `model.observe(...)`.

### Optional live calls
Set `PROVIDER="openai"` or `"together"` in S1, and set `MODEL` to a tool-capable model available to your account. Rerun S1 and the workflow. Keys use a hidden prompt; never save them in cells. Calls send fictional documents to the selected provider and are billed to your account.

**Expected live behavior is not guaranteed.** Correctness, source coverage, tool execution and agreement with tool results must be checked separately. The offline answers above refer to the scripted path.

[OpenAI function calling](https://developers.openai.com/api/docs/guides/function-calling) · [Together function calling](https://docs.together.ai/docs/function-calling)

## Continue: Harbor Metals tools, MCP and permissions
Open [01b_harbor_tools_mcp_permissions.ipynb](01b_harbor_tools_mcp_permissions.ipynb).

Try direct function calls first. Then see how the same tools can be exposed through MCP and how research, reviewer and revoked permissions change allowed actions. The core exercise runs without an API key. Optional live calls use OpenAI.

**Browser alternative:** in Google Colab choose **File → Open notebook → GitHub**, paste the notebook GitHub URL, then **Save a copy in Drive**. The companion notebook includes setup instructions.